# 예지보전 자료를 이용한 상태 분류

## AI4I 설비자료의 고장 여부 분류

센서·운전 변수를 이용해 고장 분류점수를 산출하고, confusion matrix를 실제 점검 의사결정의 오류로 번역합니다.


## 이번 교시의 학습목표

수업이 끝나면 다음을 수행할 수 있습니다.

- 회귀와 분류, 분류점수와 예측 class를 구분합니다.
- Logistic Regression과 Decision Tree를 같은 split에서 비교합니다.
- TP·FP·FN·TN을 설비 고장과 점검 상황으로 해석하고 label leakage를 점검합니다.


## 이번 교시에서 사용할 데이터

### 강의 시연: UCI AI4I 2020 Predictive Maintenance

| 항목 | 내용 |
|---|---|
| 자료 성격 | 실제 milling 공정을 참고해 만든 합성 예지보전 자료 |
| 규모 | 10,000행, 14열 |
| 한 행의 의미 | 한 제품 처리 시점의 설비·공정 상태 |
| 입력 후보 | 제품유형, 공기·공정온도, 회전속도, torque, tool wear |
| target | `Machine failure` |
| class 분포 | 정상 9,661행, 고장 339행(3.39%) |

`UDI`와 `Product ID`는 식별자입니다. `TWF`, `HDF`, `PWF`, `OSF`, `RNF`는 `Machine failure`를 구성하는 failure-mode 결과 label이므로 feature에 넣지 않습니다. 합성자료에서 얻은 성능과 비용을 실제 설비에 그대로 일반화할 수 없습니다.

### 적용 실습: Bike 고수요 분류

시간별 `cnt`를 train 자료에서 정한 기준으로 고수요/일반수요 event로 바꿉니다. cutoff를 전체 자료에서 계산하면 test 분포를 미리 보는 누수가 생길 수 있습니다. 이번 실습은 같은 시간대의 달력·날씨 조건으로 같은 시간대의 고수요 상태를 구분하므로, FN은 실제 고수요를 일반수요로 분류한 행이고 FP는 실제 일반수요를 고수요로 분류한 행입니다. 사전 재배치 경보로 확장하려면 예측 기준시점·선행시간과 그때 이용 가능한 입력정보를 다시 정의해야 합니다.


## 연속값 예측과 사건 분류

**Python 분석 초점:** 예지보전 도메인 전체를 설명하는 대신, 연속형 분류점수를 임계값으로 class로 바꾸고 오류를 표와 그림으로 확인하는 Python 절차를 다룹니다.

| 분석 문제 | 결과변수의 예 | 모형의 출력 |
|---|---|---|
| 회귀 | 전력사용량, 대여건수 | 연속적인 수치 |
| 분류 | 고장 여부, 고수요 여부 | 사건에 속할 점수와 예측분류 |

앞 교시의 회귀모형은 `kWh`처럼 연속적인 값을 예측했습니다. 이번 교시의 두 분류모형은 각 관측이 고장에 속할 점수를 산출합니다. 이 점수에 임계값을 적용해야 정상 또는 고장 경보가 결정됩니다.

$$
\hat p(y=1\mid X)
\xrightarrow{\text{임계값}}
\hat y\in\{0,1\}
$$

모형이 산출한 점수와 현장에서 사용할 경보 기준은 서로 다른 결정입니다.


## 예측점수와 임계값

임계값을 0.50으로 정한 경우를 네 관측으로 확인합니다.

| 설비 관측 | 모형 점수 | `점수 ≥ 0.50` | 예측분류 |
|---|---:|---|---|
| A | 0.12 | 아니요 | 정상(0) |
| B | 0.48 | 아니요 | 정상(0) |
| C | 0.51 | 예 | 고장 경보(1) |
| D | 0.86 | 예 | 고장 경보(1) |

B와 C의 점수 차이는 0.03에 불과하지만 임계값을 사이에 두고 있어 경보 결과는 달라집니다. 임계값은 점수를 바꾸지 않고 점수를 행동으로 변환합니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

scores = np.array([0.12, 0.48, 0.51, 0.86])
threshold = 0.50
decision = pd.DataFrame({
    "관측": list("ABCD"),
    "분류점수": scores,
    "threshold": threshold,
    "예측 class": (scores >= threshold).astype(int),
})
print("[같은 점수를 0·1 class로 바꾸기]")
display(decision)

fig, ax = plt.subplots(figsize=(8, 2.2))
ax.scatter(scores, np.zeros_like(scores), s=110, c=decision["예측 class"], cmap="coolwarm", vmin=0, vmax=1)
ax.axvline(threshold, color="black", linestyle="--", label="threshold = 0.50")
for label, score in zip(decision["관측"], scores):
    ax.text(score, 0.035, f"{label}: {score:.2f}", ha="center")
ax.set(xlim=(0, 1), yticks=[], xlabel="Classification score", title="Score and threshold")
ax.legend(); plt.tight_layout(); plt.show()
print("0.48과 0.51의 점수 차이는 작지만 threshold의 서로 다른 쪽에 있어 class가 달라집니다.")


## AI4I 2020 자료와 분석 범위

AI4I 2020은 산업 고장 분류 연구를 위해 실제 밀링 공정의 원리를 참고하여 만든 **합성자료**입니다.

| 항목 | 수업에서 사용할 정의 |
|---|---|
| 관측 수 | 10,000행 |
| 한 행 | 한 제품 처리 시점의 설비·공정 상태 |
| 결과변수 | `Machine failure` |
| 입력 후보 | 제품유형, 공기·공정온도, 회전속도, torque, tool wear |
| 양성 사건 | 고장 339건, 3.39% |

`TWF`, `HDF`, `PWF`, `OSF`, `RNF`는 고장 결과를 구성하는 failure-mode 표지이므로 입력변수에서 제외합니다. 이 자료는 분류 절차를 비교하는 데 사용하며, 특정 공장의 실제 고장률이나 정비비용을 추정하는 자료로 해석하지 않습니다.

출처: [Matzka (2020)](https://doi.org/10.1109/AI4I49448.2020.00023), [UCI AI4I 2020](https://doi.org/10.24432/C5HS5C)


## 로지스틱 회귀의 0–1 점수 변환

로지스틱 회귀는 입력변수의 선형결합을 로그 오즈와 연결합니다.

$$
\log\left(\frac{p}{1-p}\right)
=z
=\beta_0+\beta_1x_1+\cdots+\beta_px_p
$$

로지스틱 함수를 적용하면 $z$는 0과 1 사이의 값으로 변환됩니다.

$$
p=\frac{1}{1+e^{-z}}
$$

$z=0$일 때 $p=0.5$이며, $z$가 커질수록 양성 사건의 점수도 커집니다. 계수는 확률의 단순 증가량이 아니라 다른 입력이 같을 때의 로그 오즈 변화로 해석합니다.

이번 실습은 희귀 사건에 균형 가중치를 적용하므로 0–1 출력값을 실제 고장위험 확률로 직접 환산하지 않습니다. 위험확률로 해석하려면 별도의 calibration 검증이 필요합니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

linear_value = np.array([-4, -2, 0, 2, 4], dtype=float)
score = 1 / (1 + np.exp(-linear_value))
sigmoid_table = pd.DataFrame({"선형결합 z": linear_value, "sigmoid(z)": score})
print("[선형결합을 0과 1 사이 점수로 변환]")
display(sigmoid_table.round(3))

x = np.linspace(-6, 6, 300)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot(x, 1 / (1 + np.exp(-x)), linewidth=2.5)
ax.scatter(linear_value, score, color="black", zorder=3)
ax.axhline(0.5, color="gray", linestyle="--")
ax.axvline(0, color="gray", linestyle=":")
ax.set(xlabel="Linear combination z", ylabel="Classification score", title="Logistic transformation")
plt.tight_layout(); plt.show()
print("z=0이면 점수는 0.5입니다. 현재 수업의 가중 모형 점수를 보정된 실제 확률로 단정하지 않습니다.")


## 의사결정나무의 조건 분할

의사결정나무는 입력공간을 조건에 따라 반복해서 나눕니다. 이번 실습에서는 클래스 가중치가 적용되므로 말단노드의 단순 고장 비율이 아니라 **가중된 클래스 비율**이 0–1 점수로 사용됩니다.

```text
torque > 55?
├─ 아니요 → tool wear > 180?
│            ├─ 아니요 → 고장 점수 0.03
│            └─ 예     → 고장 점수 0.28
└─ 예     → 고장 점수 0.74
```

위 기준값은 작동 원리를 설명하기 위한 예시이며 실제 적합 결과가 아닙니다. 나무가 지나치게 깊거나 말단노드의 표본이 너무 적으면 학습자료의 우연한 패턴에 과도하게 적합될 수 있습니다.

두 모형은 같은 입력변수, 같은 train/test 표본, 같은 임계값에서 비교합니다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

tree_example = pd.DataFrame({
    "torque": [35, 42, 48, 58, 63, 70],
    "tool_wear": [80, 210, 120, 150, 205, 240],
})
tree_example["leaf"] = "low-score leaf"
tree_example.loc[(tree_example["torque"] > 55) & (tree_example["tool_wear"] <= 180), "leaf"] = "middle-score leaf"
tree_example.loc[(tree_example["torque"] > 55) & (tree_example["tool_wear"] > 180), "leaf"] = "high-score leaf"
score_map = {"low-score leaf": 0.08, "middle-score leaf": 0.35, "high-score leaf": 0.78}
tree_example["분류점수"] = tree_example["leaf"].map(score_map)
print("[조건을 따라 도착한 leaf에서 점수 읽기]")
display(tree_example)

fig, ax = plt.subplots(figsize=(7, 4))
scatter = ax.scatter(tree_example["torque"], tree_example["tool_wear"], c=tree_example["분류점수"], cmap="viridis", s=110)
ax.axvline(55, color="black", linestyle="--", label="torque split")
ax.hlines(180, 55, 75, color="gray", linestyle=":", label="wear split after torque > 55")
ax.set(xlim=(30, 75), xlabel="Torque", ylabel="Tool wear", title="Illustrative tree partitions")
ax.legend(); fig.colorbar(scatter, ax=ax, label="Classification score")
plt.tight_layout(); plt.show()
print("경계값은 작동원리를 보여 주는 예시이며 실제 AI4I 적합 결과가 아닙니다.")


## 혼동행렬과 현장 오류

혼동행렬은 실제 상태를 행에, 예측 상태를 열에 놓습니다.

| 실제 \ 예측 | 정상(0) | 고장 경보(1) |
|---|---:|---:|
| 정상(0) | TN = 89 | FP = 6 |
| 고장(1) | FN = 1 | TP = 4 |

이 예에서는 경보가 10건 울렸고 그중 4건이 실제 고장입니다. 실제 고장 5건 가운데 4건을 탐지하고 1건을 놓쳤습니다.

- **FP 6건:** 정상 설비에 발생한 불필요한 점검 경보
- **FN 1건:** 경보 없이 지나간 실제 고장
- **Precision 0.40:** 경보 10건 중 실제 고장 4건
- **Recall 0.80:** 실제 고장 5건 중 탐지한 고장 4건

어느 오류를 우선 줄일지는 정비비용, 안전기준, 점검 인력에 따라 달라집니다.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

cm = np.array([[89, 6], [1, 4]])
cm_table = pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"])
tn, fp, fn, tp = cm.ravel()
precision = tp / (tp + fp)
recall = tp / (tp + fn)
print("[네 종류의 분류결과와 두 지표]")
display(cm_table)
print(f"precision = {tp}/({tp}+{fp}) = {precision:.2f}")
print(f"recall    = {tp}/({tp}+{fn}) = {recall:.2f}")

fig, ax = plt.subplots(figsize=(5.2, 4.2))
sns.heatmap(cm_table, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_title("Confusion matrix: rows=actual, columns=predicted")
plt.tight_layout(); plt.show()
print("AI4I에서는 FN을 놓친 고장으로 읽습니다. Bike 적용은 미래경보가 아니라 같은 시간대 상태 오분류로 읽습니다.")


## 결과정보 누수 점검

분석 시점에 알 수 없는 결과정보가 입력변수에 포함되면 test 성능이 과대평가됩니다.

| 자료 | 누수가 되는 입력 | 이유 |
|---|---|---|
| AI4I | `TWF`, `HDF`, `PWF`, `OSF`, `RNF` | `Machine failure`를 구성하는 결과 표지 |
| Bike | `casual`, `registered` | 두 열의 합이 결과변수 `cnt` |
| 사건 기준 | 전체 자료에서 계산한 cutoff | test 결과분포를 미리 사용 |

사건 기준과 전처리 규칙은 train에서 정하고 test에는 같은 규칙만 적용합니다.


## 분류 실험의 설계와 산출물

1. 한 행과 양성 사건을 정의합니다.
2. 클래스별 관측 수와 사건 비율을 확인합니다.
3. 결과정보와 식별자를 입력변수에서 제외합니다.
4. 두 모형을 같은 train/test 표본에서 적합합니다.
5. 같은 임계값으로 예측분류를 만듭니다.
6. 혼동행렬의 FP와 FN을 현장 상황으로 해석합니다.
7. Bike 자료에서는 train의 분위수로 고수요 사건을 정의해 같은 절차를 적용합니다.

이번 교시의 산출물은 두 모형의 비교표, 혼동행렬, 그리고 두 오류의 현장 해석입니다.


## 실행 환경과 입력자료

AI4I 자료를 이용해 **현재 관측의 상태를 분류**합니다. 미래 고장을 미리 경보하는 실험과 구분합니다.

| 확인 항목 | 이번 분석의 기준 |
|---|---|
| positive class | `Machine failure = 1` |
| 입력 | 온도·회전속도·torque·tool wear와 type |
| 제외 | `UDI`, `Product ID`, target과 failure-mode 결과열 |
| 비교 | 같은 평가행·임계값의 Logistic Regression과 Tree |
| 오류 해석 | FP는 불필요 점검, FN은 놓친 고장 상태 |

다음 셀에서 사건 수, 전체 스키마, 제외열과 실제 feature 목록을 함께 확인합니다.

> **실행 전 확인:** 식별자나 failure-mode 열이 입력에 들어가면 왜 평가가 낙관적으로 변할 수 있습니까?


In [ ]:
from pathlib import Path
import platform
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)
warnings.filterwarnings("ignore", category=FutureWarning)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행하십시오."
    )
print("Python:", platform.python_version())
print("pandas:", pd.__version__)


## 1. 양성 사건과 평가표본의 정의

| 항목 | 이번 시연의 기준 |
|---|---|
| 한 행 | 한 제품 처리 시점의 센서·운전 관측 |
| 양성 사건 | `Machine failure = 1` |
| 모형 출력 | 고장 점수와 임계값 기반 경보 |
| 평가단위 | 학습에 사용하지 않은 test 행 |

모형을 적합하기 전에 클래스별 관측 수와 고장 비율을 확인합니다. `UDI`, `Product ID`와 failure-mode 결과열은 입력에서 제외하고, train과 test 모두에 정상과 고장 관측이 포함되도록 분할합니다.

> **확인 질문:** 이 분석에서 false negative는 현장에서 어떤 상황입니까?

### 이번 AI4I 시연이 답하는 시간 질문

이번 시연은 **한 제품 처리 관측에서 기록된 센서·운전값으로 같은 관측의 고장 label을 구분하는 동일 관측 상태 분류**입니다. ‘고장 몇 분 전 경보’의 성능을 측정하는 조기경보 실험은 아닙니다. 예지보전 연구로 확장하려면 예측 기준시점, 경보 선행시간, 각 센서값이 고장 전에 확보되는 시점을 명시하고 시간 또는 설비 단위의 독립 평가를 다시 설계해야 합니다.

`UDI`와 `Product ID`는 식별자이고, `TWF`, `HDF`, `PWF`, `OSF`, `RNF`는 고장 결과를 구성하는 failure-mode label이므로 입력에서 제외합니다.


In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

data = pd.read_csv(find_course_data("ai4i/ai4i2020.csv"))
target_name = "Machine failure"
numeric_features = ["Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]
categorical_features = ["Type"]
feature_columns = numeric_features + categorical_features
analysis_name = "AI4I machine-failure classification"
TASK_SCOPE = "same_record_machine_state_classification"
TASK_SCOPE_DESCRIPTION = "같은 제품 처리 관측의 센서·운전값으로 같은 관측의 고장 상태를 구분"
IDENTIFIER_COLUMNS = ["UDI", "Product ID"]
DIRECT_RESULT_COLUMNS = ["Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"]
input_timing = pd.DataFrame({
    "정보 묶음": ["공정 센서·운전값", "식별자", "고장 및 failure-mode label"],
    "이번 모형에서의 역할": ["입력", "제외", "결과 또는 결과 파생정보이므로 제외"],
    "시간 해석": ["고장 label과 같은 관측", "예측정보가 아님", "미래 경보 시점에는 사용할 수 없음"],
})
print("분석 시간범위:", TASK_SCOPE_DESCRIPTION)
display(input_timing)
print("이 결과만으로 고장 선행시간 또는 조기경보 성능을 주장할 수 없습니다.")

# UDI·Product ID는 식별자, TWF/HDF/PWF/OSF/RNF는 결과 failure-mode label이므로 제외한다.
X = data[feature_columns]
y = data[target_name]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("prevalence:", y.mean())
print("train/test positives:", int(y_train.sum()), int(y_test.sum()))


### 출력 해석: 정상과 고장 class

고장이 얼마나 희귀한지와 고장 행의 센서값 예시를 확인합니다.


In [ ]:
class_balance = y.value_counts().rename_axis("class").to_frame("rows")
class_balance["percent"] = (class_balance["rows"] / len(y) * 100).round(2)
display(class_balance)
print("고장 행 예시")
display(data.loc[data[target_name].eq(1), [target_name, *feature_columns]].head(5))


## 2. 같은 평가조건에서 두 분류모형 비교하기

| 비교 항목 | Logistic Regression | Decision Tree |
|---|---|---|
| 점수 생성 | 입력변수의 선형결합과 logistic 변환 | 조건 분할과 말단구간의 가중 사건비율 |
| 표현 가능한 관계 | 선형적인 log-odds 관계 | 비선형 조건과 상호작용 |
| 주요 과적합 통제 | 규제와 반복수렴 | 최대 깊이와 최소 말단표본 |

정상 97건과 고장 3건이 있는 학습자료를 생각해 보겠습니다. 아무 조정 없이 학습하면 다수인 정상의 오류가 학습을 지배할 수 있습니다. `class_weight="balanced"`는 희귀한 고장 행의 오분류를 더 크게 반영해 두 class가 학습에 기여하는 비중을 조정합니다.

- 고장 행을 새로 합성하거나 복제하는 기능이 아닙니다.
- test 표본의 실제 고장비율을 바꾸지 않습니다.
- 가중치를 사용한 0–1 출력은 바로 실제 고장위험 확률이라고 부르지 않습니다.
- 0.5도 자동으로 최적 운영 임계값이 되지 않습니다.

두 모형은 같은 입력변수, 학습·평가행과 가중치 조건에서 비교합니다. 상세 혼동행렬은 test 결과를 보기 전에 정한 Logistic Regression을 기준으로 읽으며, 실제 위험확률로 해석하려면 별도의 calibration 검증이 필요합니다.


In [ ]:
preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
])
models = {
    "logistic": Pipeline([("prep", clone(preprocess)), ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))]),
    "decision tree": Pipeline([("prep", clone(preprocess)), ("model", DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, class_weight="balanced", random_state=42))]),
}
for model in models.values():
    model.fit(X_train, y_train)


### 출력 해석: 분류점수가 class로 바뀌는 기준과 두 모델

네 개 분류점수를 0.5에서 class로 바꾼 뒤 같은 train 자료를 사용한 두 분류기를 확인합니다.


In [ ]:
threshold_example = pd.DataFrame({"failure_score": [0.12, 0.48, 0.51, 0.86]})
threshold_example["predicted_at_0.5"] = (threshold_example["failure_score"] >= 0.5).astype(int)
print("[같은 분류점수도 threshold의 어느 쪽에 있는지에 따라 경보가 달라집니다]")
display(threshold_example)

model_view = pd.DataFrame({
    "model": list(models),
    "score construction": ["weighted sensor score → sigmoid → 0–1 score", "conditional splits → weighted leaf score"],
    "training rows": [len(X_train)] * len(models),
})
display(model_view)


## 3. 예측점수에서 혼동행렬까지

`입력변수 → 모형 점수 → 임계값 → 예측분류 → 혼동행렬`

임계값 0.50에서 점수 `[0.12, 0.48, 0.51, 0.86]`은 분류 `[0, 0, 1, 1]`로 바뀝니다. 실제 test 출력에서는 0.5에 가까운 관측과 점수가 높은 관측을 먼저 확인한 뒤 전체 혼동행렬을 읽습니다.

| 실제 \ 예측 | 정상(0) | 고장 경보(1) |
|---|---|---|
| 정상(0) | TN | FP: 불필요한 점검 |
| 고장(1) | FN: 놓친 고장 | TP |

> **확인 질문:** 임계값을 낮출 때 경보 수, FP, FN은 일반적으로 어느 방향으로 변합니까?


In [ ]:
REFERENCE_MODEL = "logistic"  # test 결과를 보기 전에 시연용으로 고정
rows, confusion_tables, classification_scores = [], {}, {}
for name, model in models.items():
    classification_score = model.predict_proba(X_test)[:, 1]
    prediction = (classification_score >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, prediction, labels=[0, 1]).ravel()
    rows.append({
        "model": name, "accuracy": accuracy_score(y_test, prediction),
        "precision": precision_score(y_test, prediction, zero_division=0),
        "recall": recall_score(y_test, prediction, zero_division=0),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
    })
    confusion_tables[name] = pd.DataFrame([[tn, fp], [fn, tp]], index=["actual 0", "actual 1"], columns=["pred 0", "pred 1"])
    classification_scores[name] = classification_score
result_table = pd.DataFrame(rows).sort_values("model").reset_index(drop=True)
display(result_table)
print("상세 오류표는 평가점수를 보기 전에 정한 Logistic Regression을 기준으로 읽습니다.")
classification_score = classification_scores[REFERENCE_MODEL]
display(confusion_tables[REFERENCE_MODEL])


### 출력 해석: 분류점수에서 class로

분류점수가 0.5 이상이면 예측 class가 1이 되는 과정을 실제 행으로 확인합니다.


In [ ]:
classification_all = X_test.copy()
classification_all["actual"] = y_test.to_numpy()
classification_all["failure_score"] = classification_score
classification_all["predicted_at_0.5"] = (classification_score >= 0.5).astype(int)
classification_all["distance_from_0.5"] = np.abs(classification_score - 0.5)

near_threshold = classification_all.nsmallest(4, "distance_from_0.5")
highest_score = classification_all.nlargest(4, "failure_score")
classification_preview = pd.concat([near_threshold, highest_score]).loc[lambda frame: ~frame.index.duplicated()]
print("[0.5 근처 행과 고장 분류점수가 높은 행]")
display(classification_preview[["actual", "failure_score", "predicted_at_0.5", *feature_columns[:3]]].round(3))
closest = near_threshold.iloc[0]
print(
    f"0.5에 가장 가까운 행: 분류점수 {closest['failure_score']:.3f} → "
    f"예측 class {int(closest['predicted_at_0.5'])}, 실제 class {int(closest['actual'])}"
)

reference_result = result_table.loc[result_table["model"].eq(REFERENCE_MODEL)].iloc[0]
print(f"threshold 0.5에서 경보 수 = FP + TP = {int(reference_result.FP)} + {int(reference_result.TP)}")
print(f"precision = TP / (TP + FP) = {reference_result.precision:.3f}")
print(f"recall    = TP / (TP + FN) = {reference_result.recall:.3f}")

import matplotlib.pyplot as plt
import seaborn as sns
fig, ax = plt.subplots(figsize=(4.8, 3.8))
sns.heatmap(confusion_tables[REFERENCE_MODEL], annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set(title=f"Confusion matrix — {REFERENCE_MODEL}")
plt.tight_layout()
plt.show()


## 4. 분류분석의 유효성 점검

- train과 test에 정상·고장 클래스가 모두 존재합니다.
- 모형 점수는 0과 1 사이의 유한한 값입니다.
- 점수, 예측분류, 실제분류의 행 수가 같습니다.
- 식별자와 고장결과 파생열을 입력에서 제외했습니다.
- 두 모형은 같은 test 행과 임계값으로 비교했습니다.
- 혼동행렬의 실제·예측 축을 명시했습니다.

이 검사는 코드와 비교조건의 일관성을 확인합니다. 임계값 0.5의 운영 타당성, 실제 공장의 대표성, 인과관계는 별도의 검토 대상입니다.


In [ ]:
checks = pd.Series({
    "task scope declared": TASK_SCOPE in {"same_hour_state_classification", "same_record_machine_state_classification"},
    "identifiers excluded": not (set(IDENTIFIER_COLUMNS) & set(feature_columns)),
    "direct result columns excluded": not (set(DIRECT_RESULT_COLUMNS) & set(feature_columns)),
    "binary train target": set(y_train.unique()) == {0, 1},
    "binary test target": set(y_test.unique()) == {0, 1},
    "classification score range": ((classification_score >= 0) & (classification_score <= 1)).all(),
    "same prediction length": len(classification_score) == len(y_test),
    "two comparable models": len(result_table) == 2,
    "finite metrics": np.isfinite(result_table[["accuracy", "precision", "recall"]]).all().all(),
}, name="passed")
display(checks.to_frame())
assert checks.all()
print("분류 실험조건 검증: PASS")


## ChatGPT 저장 응답 검증: 분류점수와 오류를 현장 언어로 바꾸기

두 모형을 같은 조건에서 비교하되, 상세 오류표는 사전에 정한 Logistic Regression을 기준으로 읽습니다. 평가자료의 recall을 보고 기준모형을 뒤늦게 선택하지 않습니다.

ChatGPT에는 TP·FP·FN·TN을 점검 행동으로 번역하게 하고, 결과 파생열 누수와 0–1 분류점수를 실제 고장위험 확률로 오해한 표현을 찾게 합니다.


In [ ]:
review_prompt = f"""
다음 이진 분류실험을 검토해 주세요.

- 분석: {analysis_name}
- 시간 범위: {TASK_SCOPE_DESCRIPTION}
- 입력 시점 표:\n{input_timing.to_string(index=False)}
- 원자료 전체 열: {list(data.columns)}
- 결과변수: {target_name}
- 실제 입력변수: {feature_columns}
- 식별자 제외열: ['UDI', 'Product ID']
- 결과 파생 제외열: ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
- 평가자료 양성 비율: {y_test.mean():.4f}
- 임계값 0.5 결과:\n{result_table.to_string(index=False)}
- 상세 오류표 기준모형: {REFERENCE_MODEL}

아래 순서로 답해 주세요.
1. 동일 관측 상태 분류와 조기경보를 구분했는지, 실제 입력변수의 결과정보 누수와 제외열 확인
2. TP·FP·FN·TN의 현장 의미
3. 정확도만으로 비교하면 놓치는 내용
4. 분류점수와 운영 임계값을 분리한 결과문장

판정은 통과 / 수정 필요 / 정보 부족 중 하나로 표시해 주세요.
제약: 제공되지 않은 정보는 추정하지 말고, 균형 가중치가 적용된 0–1 점수를 실제 고장위험 확률로 단정하지 말고, 평가점수로 기준모형을 다시 선택하지 마세요.
""".strip()
print("[ChatGPT에 복사할 프롬프트]\n")
print(review_prompt)


### ChatGPT 저장 응답을 검증할 때

1. TP·FP·FN·TN의 분모와 현장 의미를 혼동하지 않았는지 확인합니다.
2. 결과 파생열과 식별자가 입력변수에 들어가지 않았는지 다시 봅니다.
3. 균형 가중치 모형의 점수를 실제 고장위험 확률이라고 쓴 문장은 수정합니다.


## 해석 범위

AI4I 분석은 합성 센서자료에서 고장분류 절차와 오류 구성을 비교한 결과입니다. Bike의 고수요 기준은 train 자료의 통계적 분위수로 정의했습니다.

두 결과 모두 실제 설비의 고장위험, 도시의 운영비용 또는 최적 경보정책을 직접 추정하지 않습니다. 새로운 현장·도시·기간에 적용하려면 사건 정의, 발생률, 비용과 임계값을 다시 평가해야 합니다.


## 오늘의 결론과 다음 단계

### 이번 교시의 결론

사건을 정의한 뒤 분류점수와 임계값을 구분하고, 같은 평가행의 혼동행렬에서 TP·FP·FN·TN을 실제 의사결정 언어로 읽습니다.

### 적용 순서

1. `09_predictive_maintenance_classification_follow.ipynb`에서 핵심 시연을 같은 조건으로 재현합니다.
2. `09_predictive_maintenance_classification_practice.ipynb`에서 지정된 데이터 또는 조건 하나만 바꿔 비교합니다.
3. 변경조건, 비교표 또는 그림, 결과문장과 현재 검증하지 못한 범위를 함께 기록합니다.

### 실습 중 확인할 질문

- 양성 사건과 분류점수의 임계값을 각각 어떻게 정의했습니까?
- 이 문제에서 false negative는 현실적으로 무엇을 뜻합니까?
- 결과로부터 만들어진 열이 feature에 포함되지 않았습니까?

마지막 `PASS`는 Notebook에 적힌 실행조건을 확인한 결과이며 연구결론의 타당성을 대신하지 않습니다.
